# From Cells to Organisms

_Investigation `multicellularity` — coder reproduction notebook._

**Question.** Under what physical, metabolic, ecological, developmental, and evolutionary conditions can independent cells become a persistent, integrated, self-reproducing multicellular evolutionary individual? The existence of adherent cells is not sufficient to establish a major transition; a defensible investigation must identify a SEQUENCE of separately measurable properties — collective formation, physiological coordination, reproducible group reproduction, heritable group-level traits, and adaptation at the collective level — which may evolve in different orders and need not all appear in every lineage.

Seed for a falsifiable program testing which conditions turn independent cells into an integrated, self-reproducing multicellular individual.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Morowitz emergence framework**
- **De novo experimental evolution**
- **Comparative phylogenetic origins**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-genesis/viva-genesis').is_dir():
    REPO = Path('/home/runner/work/viva-genesis/viva-genesis')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_genesis.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Collective formation (`s01-formation`)

**Question.** Which adhesion, reproductive, and ecological regimes create persistent collectives rather than transient aggregates? (Q1)

**Hypothesis.** Clonal incomplete separation with reversible adhesion, swept over bond strength and division/fragmentation, produces a persistent group life cycle and heritable cluster traits that a non-group null does not.

**Claim.** Clonal incomplete separation with reversible adhesion produces a PERSISTENT group life cycle (group fraction=0.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.cell_collective.cell_collective` | 0 | adhesion=0.8, fragmentation=0.2, predation_size_reward=0.02, transport_penalty=0.02 |

Declared parameter sets (`study.yaml` variants):

- **no-adhesion-null** — `adhesion=0.02`, `fragmentation=0.8`, `predation_size_reward=0.8`, `transport_penalty=0.02`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cell_collective.cell_collective`**

Multicellularity first-cycle model: mean-field unicell/group population under adhesion, size-dependent predation/settling ecology, and oxygen-limited growth.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cell_collective_cell_collective = {
    'adhesion'                : 0.8,   # clonal incomplete separation / bond drive
    'fragmentation'           : 0.2,   # group breakup rate
    'division_rate'           : 0.5,   # per-cell division rate
    'base_predation'          : 0.4,   # size-independent death rate
    'predation_size_reward'   : 0.8,   # ecology: group size reduces predation
    'predation_size_threshold': 3.0,   # size at which protection engages
    'transport_penalty'       : 0.8,   # oxygen-limited growth penalty for large groups
    'oxygen_size_limit'       : 6.0,   # size above which interior starves
    'size_scale'              : 4.0,   # adhesion/fragmentation -> mean group size
    'relax_rate'              : 0.4,   # relaxation rate toward equilibrium
    'sharpness'               : 2.0,   # sigmoid sharpness k
    'seed'                    : 1,   # random seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s01-formation ===
STUDY = 's01-formation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cell_collective_cell_collective if 'ov_viva_genesis_composites_cell_collective_cell_collective' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cell_collective_cell_collective = materialize_composite('viva_genesis.composites.cell_collective.cell_collective', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cell_collective_cell_collective['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.cell_collective.cell_collective) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**formation-persistence**


In [ ]:
# formation-persistence
show_viz(_render_one('local:FormationPersistenceViz', {'title': 'Clonal adhesion builds a persistent group life cycle'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| group-life-cycle-persists _(primary)_ | group_life_cycle_persistence | op >= value 0.5 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Morowitz 2002, The Emergence of Everything, ch.14 | proposed |
|  |  | Ratcliff et al. 2012, PNAS | proposed |
|  |  | Herron et al. 2019, Sci Rep | proposed |


## Study: Ecological selection (`s02-ecological-selection`)

**Question.** Can size-dependent predation and settling reward group formation without added physiology? (Q1)

**Hypothesis.** Under size-dependent predation/settling, simple adhesive collectives gain a fitness advantage over unicells even with metabolic exchange controlled (H1).

**Claim.** Under size-dependent predation/settling, groups gain a FITNESS ADVANTAGE over unicells (adv=+0.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.cell_collective.cell_collective` | 0 | adhesion=0.8, fragmentation=0.2, predation_size_reward=0.8, transport_penalty=0.02 |

Declared parameter sets (`study.yaml` variants):

- **adhesion-only-control** — `adhesion=0.8`, `fragmentation=0.2`, `predation_size_reward=0.02`, `transport_penalty=0.02`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cell_collective.cell_collective`**

Multicellularity first-cycle model: mean-field unicell/group population under adhesion, size-dependent predation/settling ecology, and oxygen-limited growth.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cell_collective_cell_collective = {
    'adhesion'                : 0.8,   # clonal incomplete separation / bond drive
    'fragmentation'           : 0.2,   # group breakup rate
    'division_rate'           : 0.5,   # per-cell division rate
    'base_predation'          : 0.4,   # size-independent death rate
    'predation_size_reward'   : 0.8,   # ecology: group size reduces predation
    'predation_size_threshold': 3.0,   # size at which protection engages
    'transport_penalty'       : 0.8,   # oxygen-limited growth penalty for large groups
    'oxygen_size_limit'       : 6.0,   # size above which interior starves
    'size_scale'              : 4.0,   # adhesion/fragmentation -> mean group size
    'relax_rate'              : 0.4,   # relaxation rate toward equilibrium
    'sharpness'               : 2.0,   # sigmoid sharpness k
    'seed'                    : 1,   # random seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s02-ecological-selection ===
STUDY = 's02-ecological-selection'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cell_collective_cell_collective if 'ov_viva_genesis_composites_cell_collective_cell_collective' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cell_collective_cell_collective = materialize_composite('viva_genesis.composites.cell_collective.cell_collective', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cell_collective_cell_collective['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.cell_collective.cell_collective) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ecological-selection**


In [ ]:
# ecological-selection
show_viz(_render_one('local:EcologicalSelectionViz', {'title': 'Ecological selection — not adhesion — makes groups fitter'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| group-fitness-advantage-under-selection _(primary)_ |  | op > value 0.0 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Herron et al. 2019, Sci Rep | proposed |
|  |  | Ratcliff et al. 2012, PNAS | proposed |
|  |  | Grosberg & Strathmann 2007, Annu Rev Ecol Evol Syst | proposed |


## Study: Transport limits (`s03-transport-limits`)

**Question.** How do oxygen, nutrients, cell shape, packing and geometry limit collective size? (Q2)

**Hypothesis.** Coupling oxygen/nutrient/waste PDEs to growth across colony geometries predicts size/viability changes that track diffusion length and metabolic regime (C3).

**Claim.** Oxygen/transport BOUNDS the advantage to an optimal intermediate size: small groups (size=3.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.cell_collective.cell_collective` | 0 | adhesion=0.8, fragmentation=0.2, predation_size_reward=0.8, transport_penalty=0.8 |

Declared parameter sets (`study.yaml` variants):

- **small-groups** — `adhesion=0.2`, `fragmentation=0.3`, `predation_size_reward=0.8`, `transport_penalty=0.8`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cell_collective.cell_collective`**

Multicellularity first-cycle model: mean-field unicell/group population under adhesion, size-dependent predation/settling ecology, and oxygen-limited growth.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cell_collective_cell_collective = {
    'adhesion'                : 0.8,   # clonal incomplete separation / bond drive
    'fragmentation'           : 0.2,   # group breakup rate
    'division_rate'           : 0.5,   # per-cell division rate
    'base_predation'          : 0.4,   # size-independent death rate
    'predation_size_reward'   : 0.8,   # ecology: group size reduces predation
    'predation_size_threshold': 3.0,   # size at which protection engages
    'transport_penalty'       : 0.8,   # oxygen-limited growth penalty for large groups
    'oxygen_size_limit'       : 6.0,   # size above which interior starves
    'size_scale'              : 4.0,   # adhesion/fragmentation -> mean group size
    'relax_rate'              : 0.4,   # relaxation rate toward equilibrium
    'sharpness'               : 2.0,   # sigmoid sharpness k
    'seed'                    : 1,   # random seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s03-transport-limits ===
STUDY = 's03-transport-limits'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cell_collective_cell_collective if 'ov_viva_genesis_composites_cell_collective_cell_collective' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cell_collective_cell_collective = materialize_composite('viva_genesis.composites.cell_collective.cell_collective', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cell_collective_cell_collective['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.cell_collective.cell_collective) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**transport-size-optimum**


In [ ]:
# transport-size-optimum
show_viz(_render_one('local:TransportSizeOptimumViz', {'title': 'Transport bounds the advantage to an intermediate size optimum'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| size-viability-tracks-transport _(primary)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Knoll 2011, Annu Rev Earth Planet Sci | proposed |
|  |  | Bozdag et al. 2023, Nature | proposed |
|  |  | Bozdag et al. 2023 data/code (GitHub) | proposed |


## Study: Morphomechanics (`s04-morphomechanics`)

**Question.** How do cell aspect ratio, branch topology, packing and bond strength set collective toughness and propagule size? (Q2)

**Hypothesis.** Varying aspect ratio, branch topology, packing and bond strength predicts toughness and propagule-size distributions; mechanics backends agree on the summary observables (C4).

**Claim.** Cell adhesion (homotypic contact energy) sets collective toughness: a clonal CPM cluster under growth-driven stress stays one cohesive propagule at strong adhesion and fractures into multiple propagules at weak adhesion. UNPROVEN until the adhesion x seed sweep is graded; aspect ratio, branch topology and cross-backend agreement (C4) are not tested here.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.multicellularity_world.s04_morphomechanics` | 40 | adhesion=0.75, seed=1, grow_rate=2.0, divide_threshold=55.0, n_steps=40 |

Declared parameter sets (`study.yaml` variants):

- **adh0p0-s1** — `adhesion=0.0`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p0-s2** — `adhesion=0.0`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p0-s3** — `adhesion=0.0`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p3-s1** — `adhesion=0.3`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p3-s2** — `adhesion=0.3`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p3-s3** — `adhesion=0.3`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p5-s1** — `adhesion=0.5`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p5-s2** — `adhesion=0.5`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p5-s3** — `adhesion=0.5`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p62-s1** — `adhesion=0.62`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p62-s2** — `adhesion=0.62`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p62-s3** — `adhesion=0.62`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p75-s2** — `adhesion=0.75`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p75-s3** — `adhesion=0.75`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p9-s1** — `adhesion=0.9`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p9-s2** — `adhesion=0.9`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p9-s3** — `adhesion=0.9`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p95-s1** — `adhesion=0.95`, `seed=1`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p95-s2** — `adhesion=0.95`, `seed=2`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`
- **adh0p95-s3** — `adhesion=0.95`, `seed=3`, `grow_rate=2.0`, `divide_threshold=55.0`, `n_steps=40`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.multicellularity_world.s04_morphomechanics`**

Multicellularity S04 (morphomechanics). A clonal cell cluster grows and divides on the Rust CPM engine; the adhesion knob sets homotypic contact energy. Strong adhesion keeps the cluster one cohesive body; weak adhesion lets growth fracture it into propagules. Emits n_propagules and largest_cluster_fraction.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_multicellularity_world_s04_morphomechanics = {
    'adhesion'        : 0.8,   # cell-cell adhesion in [0,1]; higher = lower homotypic J = cohesive
    'grow_rate'       : 2.0,   # target-volume growth per update per cell
    'divide_threshold': 55.0,   # volume at which a cell divides
    'seed'            : 1,   # Potts RNG seed (vary for replicate runs)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s04-morphomechanics ===
STUDY = 's04-morphomechanics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 40
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_multicellularity_world_s04_morphomechanics if 'ov_viva_genesis_composites_multicellularity_world_s04_morphomechanics' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_multicellularity_world_s04_morphomechanics = materialize_composite('viva_genesis.composites.multicellularity_world.s04_morphomechanics', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_multicellularity_world_s04_morphomechanics['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.multicellularity_world.s04_morphomechanics) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cluster-fracture**


In [ ]:
# cluster-fracture
show_viz(_render_one('local:ClusterFractureViz', {'title': 'Weak adhesion lets growth snap a cluster into propagules', 'weak_adhesion': 0.0, 'strong_adhesion': 0.95, 'seed': 1, 'n_steps': 40}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| propagule-toughness-predicted _(primary)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Bozdag et al. 2023, Nature | proposed |
|  |  | Bozdag et al. 2023 data/code (GitHub) | proposed |
|  |  | NCBI SRA PRJNA943273 | proposed |


## Study: Cell-state differentiation (`s05-differentiation`)

**Question.** Can environmental gradients create stable cell-state differentiation without a dedicated developmental program? (Q4, Q3)

**Hypothesis.** Metabolite exchange plus inducible cell fate yields specialization that improves out-of-sample group performance beyond a no-fate and random-fate null (C5).

**Claim.** Gradient-induced specialization can improve out-of-sample group performance beyond non-regulated nulls (C5) — provisional until a no-fate/random-fate null is beaten.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.cell_collective.cell_collective` | 0 | adhesion=0.8, fragmentation=0.2, predation_size_reward=0.02, transport_penalty=0.02 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cell_collective.cell_collective`**

Multicellularity first-cycle model: mean-field unicell/group population under adhesion, size-dependent predation/settling ecology, and oxygen-limited growth.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cell_collective_cell_collective = {
    'adhesion'                : 0.8,   # clonal incomplete separation / bond drive
    'fragmentation'           : 0.2,   # group breakup rate
    'division_rate'           : 0.5,   # per-cell division rate
    'base_predation'          : 0.4,   # size-independent death rate
    'predation_size_reward'   : 0.8,   # ecology: group size reduces predation
    'predation_size_threshold': 3.0,   # size at which protection engages
    'transport_penalty'       : 0.8,   # oxygen-limited growth penalty for large groups
    'oxygen_size_limit'       : 6.0,   # size above which interior starves
    'size_scale'              : 4.0,   # adhesion/fragmentation -> mean group size
    'relax_rate'              : 0.4,   # relaxation rate toward equilibrium
    'sharpness'               : 2.0,   # sigmoid sharpness k
    'seed'                    : 1,   # random seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s05-differentiation ===
STUDY = 's05-differentiation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cell_collective_cell_collective if 'ov_viva_genesis_composites_cell_collective_cell_collective' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cell_collective_cell_collective = materialize_composite('viva_genesis.composites.cell_collective.cell_collective', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cell_collective_cell_collective['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.cell_collective.cell_collective) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| specialization-improves-group-performance _(primary)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Kirk 2005, BioEssays | proposed |
|  |  | Grosberg & Strathmann 2007, Annu Rev Ecol Evol Syst | proposed |


## Study: Evolutionary individuality (`s06-evolutionary-individuality`)

**Question.** When do reproduction, heritability and conflict suppression shift evolutionary adaptation to the collective level? (Q5)

**Hypothesis.** Single-cell bottlenecks and multicellular propagules maintain heritable between-group fitness gains across generations better than mixed-genotype groups with cheating (C2, C6).

**Claim.** Single-cell bottlenecks / multicellular propagules sustain heritable between-group fitness better than mixed-genotype groups with cheating (C2, C6) — provisional until multigenerational multilevel-selection runs.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_genesis.composites.cell_collective.cell_collective` | 0 | adhesion=0.8, fragmentation=0.2, predation_size_reward=0.02, transport_penalty=0.02 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cell_collective.cell_collective`**

Multicellularity first-cycle model: mean-field unicell/group population under adhesion, size-dependent predation/settling ecology, and oxygen-limited growth.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cell_collective_cell_collective = {
    'adhesion'                : 0.8,   # clonal incomplete separation / bond drive
    'fragmentation'           : 0.2,   # group breakup rate
    'division_rate'           : 0.5,   # per-cell division rate
    'base_predation'          : 0.4,   # size-independent death rate
    'predation_size_reward'   : 0.8,   # ecology: group size reduces predation
    'predation_size_threshold': 3.0,   # size at which protection engages
    'transport_penalty'       : 0.8,   # oxygen-limited growth penalty for large groups
    'oxygen_size_limit'       : 6.0,   # size above which interior starves
    'size_scale'              : 4.0,   # adhesion/fragmentation -> mean group size
    'relax_rate'              : 0.4,   # relaxation rate toward equilibrium
    'sharpness'               : 2.0,   # sigmoid sharpness k
    'seed'                    : 1,   # random seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s06-evolutionary-individuality ===
STUDY = 's06-evolutionary-individuality'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cell_collective_cell_collective if 'ov_viva_genesis_composites_cell_collective_cell_collective' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cell_collective_cell_collective = materialize_composite('viva_genesis.composites.cell_collective.cell_collective', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cell_collective_cell_collective['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_genesis.composites.cell_collective.cell_collective) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| between-group-heritable-fitness-stable _(primary)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  | Grosberg & Strathmann 2007, Annu Rev Ecol Evol Syst | proposed |
|  |  | Ratcliff et al. 2012, PNAS | proposed |


## Open decisions
- Which test system anchors the first cycle — snowflake yeast (settling) or predator-selected Chlamydomonas?
- Ingest the Bozdag 2023 source data/code (R8) and SRA PRJNA943273 (R9), or treat them as comparison-only?
